# Multi-Plane Multi-Channel Imaging Example

This notebook demonstrates how to use the `ndx-microscopy` extension for a combined **multi-plane** and **multi-channel** acquisition scenario, as commonly encountered in two-photon microscopy with an electrically tunable lens (ETL).

## Experimental design

- **Multi-plane acquisition at 3 imaging depths**: −50 µm, 0 µm, +50 µm below the cortical surface
- **3 simultaneous channels per plane**:
  - **Channel 1 – GCaMP6f** (pan-neuronal activity indicator): temporal `PlanarMicroscopySeries` acquired at 30 Hz
  - **Channel 2 – mCherry** (anatomical marker, excitatory-neuron subpopulation): single `PlanarMicroscopyStaticImage` per depth
  - **Channel 3 – tdTomato** (anatomical marker, inhibitory-neuron subpopulation): single `PlanarMicroscopyStaticImage` per depth

## NWB data organisation

All imaging data are grouped into a two-level container hierarchy inside `nwbfile.acquisition`:

```
acquisition/
└── multi_plane_multi_channel_data/          # MultiChannelMicroscopyContainer
    ├── gcamp_planes/                         # MultiPlaneMicroscopyContainer (GCaMP6f)
    │   ├── gcamp_-50um/                      # PlanarMicroscopySeries
    │   ├── gcamp_0um/
    │   └── gcamp_50um/
    ├── mcherry_planes/                       # MultiPlaneMicroscopyContainer (mCherry)
    │   ├── mcherry_-50um/                    # PlanarMicroscopyStaticImage
    │   ├── mcherry_0um/
    │   └── mcherry_50um/
    └── tdtomato_planes/                      # MultiPlaneMicroscopyContainer (tdTomato)
        ├── tdtomato_-50um/                   # PlanarMicroscopyStaticImage
        ├── tdtomato_0um/
        └── tdtomato_50um/
```

ROI segmentation is performed on the GCaMP6f planes only, linking each `MicroscopyResponseSeries` to the corresponding `PlanarMicroscopySeries`.

## Imports

In [ ]:
from datetime import datetime
from uuid import uuid4

import numpy as np
from pynwb import NWBFile, NWBHDF5IO

from ndx_microscopy import (
    MicroscopeModel,
    Microscope,
    MicroscopyRig,
    MicroscopyExperimentMetadata,
    MicroscopyChannel,
    LineScan,
    PlanarImagingSpace,
    PlanarMicroscopySeries,
    PlanarMicroscopyStaticImage,
    MultiPlaneMicroscopyContainer,
    MultiChannelMicroscopyContainer,
    PlanarSegmentation,
    SummaryImage,
    SegmentationContainer,
    MicroscopyResponseSeries,
    MicroscopyResponseSeriesContainer,
)

from ndx_ophys_devices import (
    ExcitationSourceModel,
    PulsedExcitationSource,
    BandOpticalFilterModel,
    BandOpticalFilter,
    DichroicMirrorModel,
    DichroicMirror,
    PhotodetectorModel,
    Photodetector,
    Indicator,
    ViralVector,
    ViralVectorInjection,
    StereotacticPosition,
)

## 1. Create the NWB file

In [ ]:
nwbfile = NWBFile(
    session_description="Multi-plane multi-channel two-photon calcium imaging with pan-neuronal GCaMP6f "
                        "and two anatomical markers (mCherry, tdTomato) across three cortical depths.",
    identifier=str(uuid4()),
    session_start_time=datetime.now(),
    lab="Neural Circuits Lab",
    institution="University of Neuroscience",
    experiment_description="Three-depth two-photon imaging of visual cortex with pan-neuronal GCaMP6f activity "
                           "imaging and anatomical subpopulation markers.",
)

## 2. Microscope hardware

Two-photon microscope with an **electrically tunable lens (ETL)** that enables rapid focal-plane switching between depths.

In [ ]:
# Microscope model and instance
microscope_model = MicroscopeModel(
    name="etl_2p_model",
    description="Two-photon microscope model with electrically tunable lens",
    model_number="ETL-2P-001",
    manufacturer="Custom Build",
)
nwbfile.add_device_model(microscope_model)

microscope = Microscope(
    name="etl_2p_scope",
    description="Two-photon microscope with ETL for multi-plane imaging",
    serial_number="ETL-SN-001",
    model=microscope_model,
    technique="electrically tunable lens",
)
nwbfile.add_device(microscope)

In [ ]:
# Femtosecond pulsed laser (920 nm for GCaMP + mCherry/tdTomato)
laser_model = ExcitationSourceModel(
    name="chameleon_model",
    manufacturer="Coherent",
    model_number="Chameleon Ultra II",
    description="Femtosecond Ti:Sapphire pulsed laser",
    source_type="laser",
    excitation_mode="two-photon",
    wavelength_range_in_nm=[680.0, 1080.0],
)
nwbfile.add_device_model(laser_model)

laser = PulsedExcitationSource(
    name="laser",
    description="Femtosecond pulsed laser for multi-plane multi-channel two-photon imaging",
    serial_number="CU2-SN-001",
    model=laser_model,
    power_in_W=1.5,
    peak_power_in_W=100000.0,
    peak_pulse_energy_in_J=1.25e-9,
    pulse_rate_in_Hz=80.0e6,
)
nwbfile.add_device(laser)

In [ ]:
# Optical components — GCaMP6f emission path
excitation_filter_model = BandOpticalFilterModel(
    name="excitation_filter_model",
    filter_type="Bandpass",
    manufacturer="Semrock",
    model_number="FF01-920/80",
    center_wavelength_in_nm=920.0,
    bandwidth_in_nm=80.0,
)
nwbfile.add_device_model(excitation_filter_model)

excitation_filter = BandOpticalFilter(
    name="excitation_filter",
    description="Excitation bandpass filter",
    serial_number="EF-SN-001",
    model=excitation_filter_model,
)
nwbfile.add_device(excitation_filter)

dichroic_model = DichroicMirrorModel(
    name="primary_dichroic_model",
    manufacturer="Semrock",
    model_number="FF757-Di01",
    cut_on_wavelength_in_nm=757.0,
    angle_of_incidence_in_degrees=45.0,
)
nwbfile.add_device_model(dichroic_model)

dichroic = DichroicMirror(
    name="primary_dichroic",
    description="Primary dichroic mirror",
    serial_number="DM-SN-001",
    model=dichroic_model,
)
nwbfile.add_device(dichroic)

# GCaMP6f emission filter (green channel)
emission_filter_green_model = BandOpticalFilterModel(
    name="emission_filter_green_model",
    filter_type="Bandpass",
    manufacturer="Semrock",
    model_number="FF01-510/84",
    center_wavelength_in_nm=510.0,
    bandwidth_in_nm=84.0,
)
nwbfile.add_device_model(emission_filter_green_model)

emission_filter_green = BandOpticalFilter(
    name="emission_filter_green",
    description="Green emission filter for GCaMP6f",
    serial_number="EFG-SN-001",
    model=emission_filter_green_model,
)
nwbfile.add_device(emission_filter_green)

# mCherry / tdTomato emission filter (red channel)
emission_filter_red_model = BandOpticalFilterModel(
    name="emission_filter_red_model",
    filter_type="Bandpass",
    manufacturer="Semrock",
    model_number="FF01-607/70",
    center_wavelength_in_nm=607.0,
    bandwidth_in_nm=70.0,
)
nwbfile.add_device_model(emission_filter_red_model)

emission_filter_red = BandOpticalFilter(
    name="emission_filter_red",
    description="Red emission filter for mCherry / tdTomato",
    serial_number="EFR-SN-001",
    model=emission_filter_red_model,
)
nwbfile.add_device(emission_filter_red)

# GaAsP PMT detectors
pmt_model = PhotodetectorModel(
    name="pmt_model",
    detector_type="GaAsP PMT",
    manufacturer="Hamamatsu",
    model_number="H10770PA-40",
    gain=50.0,
    gain_unit="dB",
)
nwbfile.add_device_model(pmt_model)

pmt_green = Photodetector(
    name="pmt_green",
    description="GaAsP PMT for green (GCaMP6f) channel",
    serial_number="PMT-G-001",
    model=pmt_model,
)
nwbfile.add_device(pmt_green)

## 3. Microscopy rig

In [ ]:
microscopy_rig = MicroscopyRig(
    name="etl_2p_rig",
    description="Multi-plane two-photon rig with ETL; simultaneous green (GCaMP6f) and red (mCherry/tdTomato) channels",
    microscope=microscope,
    excitation_source=laser,
    excitation_filter=excitation_filter,
    dichroic_mirror=dichroic,
    emission_filter=emission_filter_green,  # primary detector path
    photodetector=pmt_green,
)

## 4. Biological indicators and experiment metadata

Three indicators are used:
| Indicator | Purpose | Subpopulation |
|-----------|---------|---------------|
| GCaMP6f   | Calcium activity (pan-neuronal) | all neurons |
| mCherry   | Anatomical marker | excitatory (CaMKII+) neurons |
| tdTomato  | Anatomical marker | inhibitory (PV+) neurons |

In [ ]:
# Viral vectors
vv_gcamp = ViralVector(
    name="aav_gcamp",
    construct_name="AAV1-Syn-GCaMP6f-WPRE-SV40",
    manufacturer="Addgene",
    titer_in_vg_per_ml=1.0e13,
    description="Pan-neuronal GCaMP6f driven by synapsin promoter",
)
vv_mcherry = ViralVector(
    name="aav_mcherry",
    construct_name="AAV5-CaMKII-mCherry",
    manufacturer="Addgene",
    titer_in_vg_per_ml=5.0e12,
    description="mCherry in excitatory neurons (CaMKII promoter)",
)
vv_tdtomato = ViralVector(
    name="aav_tdtomato",
    construct_name="AAV5-DIO-tdTomato",
    manufacturer="Addgene",
    titer_in_vg_per_ml=5.0e12,
    description="tdTomato in PV+ inhibitory neurons (Cre-dependent)",
)

# Injections
viral_injection_coordinates = StereotacticPosition(
    name="viral_injection_coordinates",
    anatomical_target="Visual cortex",
    origin= "bregma",
    orientation="RAS",
    x_in_mm = 1.5,
    y_in_mm  = -3.0,
    z_in_mm= 0.0,
    pitch_in_deg = 180.0,
    yaw_in_deg = 90.0,
    roll_in_deg = 0.0,
    )
inj_gcamp = ViralVectorInjection(
    name="injection_gcamp",
    volume_in_uL=0.5,
    injection_date="2024-01-10T12:00:00+00:00",
    viral_injection_coordinates=viral_injection_coordinates,
    viral_vector=vv_gcamp,
)
inj_mcherry = ViralVectorInjection(
    name="injection_mcherry",
    volume_in_uL=0.3,
    injection_date="2024-01-10T12:00:00+00:00",
    viral_injection_coordinates=viral_injection_coordinates,
    viral_vector=vv_mcherry,
)
inj_tdtomato = ViralVectorInjection(
    name="injection_tdtomato",
    volume_in_uL=0.3,
    viral_injection_coordinates=viral_injection_coordinates,
    injection_date="2024-01-10T12:00:00+00:00",
    viral_vector=vv_tdtomato,
)

# Indicators
indicator_gcamp = Indicator(
    name="GCaMP6f",
    label="GCaMP6f",
    description="Pan-neuronal genetically encoded calcium indicator",
    manufacturer="Addgene",
    viral_vector_injection=inj_gcamp,
)
indicator_mcherry = Indicator(
    name="mCherry",
    label="mCherry",
    description="Anatomical marker for CaMKII+ excitatory neurons",
    manufacturer="Addgene",
    viral_vector_injection=inj_mcherry,
)
indicator_tdtomato = Indicator(
    name="tdTomato",
    label="tdTomato",
    description="Anatomical marker for PV+ inhibitory neurons (Cre-dependent)",
    manufacturer="Addgene",
    viral_vector_injection=inj_tdtomato,
)

# Centralised experiment metadata
microscopy_experiment_metadata = MicroscopyExperimentMetadata(
    viral_vectors=[vv_gcamp, vv_mcherry, vv_tdtomato],
    viral_vector_injections=[inj_gcamp, inj_mcherry, inj_tdtomato],
    indicators=[indicator_gcamp, indicator_mcherry, indicator_tdtomato],
    microscopy_rigs=[microscopy_rig],
)
nwbfile.add_lab_meta_data(microscopy_experiment_metadata)

## 5. Microscopy channels

Each channel is defined by its excitation/emission wavelengths and linked to the corresponding indicator.

In [ ]:
channel_gcamp = MicroscopyChannel(
    name="gcamp_channel",
    description="Pan-neuronal GCaMP6f — two-photon excitation at 920 nm, green emission",
    excitation_wavelength_in_nm=920.0,
    emission_wavelength_in_nm=510.0,
    indicator=indicator_gcamp,
)

channel_mcherry = MicroscopyChannel(
    name="mcherry_channel",
    description="mCherry anatomical label — two-photon excitation at 1040 nm, red emission",
    excitation_wavelength_in_nm=1040.0,
    emission_wavelength_in_nm=610.0,
    indicator=indicator_mcherry,
)

channel_tdtomato = MicroscopyChannel(
    name="tdtomato_channel",
    description="tdTomato anatomical label (PV+ inhibitory) — two-photon excitation at 1040 nm, red emission",
    excitation_wavelength_in_nm=1040.0,
    emission_wavelength_in_nm=581.0,
    indicator=indicator_tdtomato,
)

## 6. Illumination pattern

In [ ]:
# Single line-scan pattern shared by all imaging spaces
line_scan = LineScan(
    name="line_scan",
    description="Raster line scanning two-photon microscopy",
    scan_direction="horizontal",
    line_rate_in_Hz=1000.0,
    dwell_time_in_s=1.0e-6,
)

## 7. Per-depth imaging data

For each of the three depths we create:
- One `PlanarImagingSpace` shared by all channels at that depth
- One `PlanarMicroscopySeries` for GCaMP6f (functional, 30 Hz)
- One `PlanarMicroscopyStaticImage` for mCherry (anatomical)
- One `PlanarMicroscopyStaticImage` for tdTomato (anatomical)

In [ ]:
depths_in_um = [-50, 0, 50]   # focal depths relative to cortical surface
frames = 1000                  # number of time points for functional channel
height, width = 256, 256       # image dimensions (pixels)
frame_rate_hz = 30.0

gcamp_series_list   = []
mcherry_images_list = []
tdtomato_images_list = []

for depth in depths_in_um:
    # ----- Imaging space (shared across channels for this depth) -----
    # Each channel gets its own PlanarImagingSpace object
    # so that the illumination_pattern link is unambiguous.

    space_gcamp = PlanarImagingSpace(
        name=f"plane_gcamp_{depth}um",
        description=f"GCaMP6f imaging plane at {depth} µm relative to cortical surface",
        pixel_size_in_um=[0.8, 0.8],
        dimensions_in_pixels=[height, width],
        anatomical_target="Primary visual cortex (V1)",
        illumination_pattern=line_scan,
    )

    space_mcherry = PlanarImagingSpace(
        name=f"plane_mcherry_{depth}um",
        description=f"mCherry imaging plane at {depth} µm relative to cortical surface",
        pixel_size_in_um=[0.8, 0.8],
        dimensions_in_pixels=[height, width],
        anatomical_target="Primary visual cortex (V1)",
        illumination_pattern=line_scan,
    )

    space_tdtomato = PlanarImagingSpace(
        name=f"plane_tdtomato_{depth}um",
        description=f"tdTomato imaging plane at {depth} µm relative to cortical surface",
        pixel_size_in_um=[0.8, 0.8],
        dimensions_in_pixels=[height, width],
        anatomical_target="Primary visual cortex (V1)",
        illumination_pattern=line_scan,
    )

    # ----- Channel 1: GCaMP6f (functional, PlanarMicroscopySeries) -----
    data_gcamp = np.random.rand(frames, height, width).astype(np.float32)
    gcamp_series = PlanarMicroscopySeries(
        name=f"gcamp_{depth}um",
        description=f"GCaMP6f calcium imaging at {depth} µm",
        microscopy_rig=microscopy_rig,
        microscopy_channel=channel_gcamp,
        planar_imaging_space=space_gcamp,
        data=data_gcamp,
        unit="a.u.",
        rate=frame_rate_hz,
        starting_time=0.0,
    )
    gcamp_series_list.append(gcamp_series)

    # ----- Channel 2: mCherry (anatomical, PlanarMicroscopyStaticImage) -----
    data_mcherry = np.random.rand(height, width).astype(np.float32)
    mcherry_image = PlanarMicroscopyStaticImage(
        name=f"mcherry_{depth}um",
        description=f"mCherry anatomical image at {depth} µm — excitatory (CaMKII+) neurons",
        microscopy_rig=microscopy_rig,
        microscopy_channel=channel_mcherry,
        planar_imaging_space=space_mcherry,
        data=data_mcherry,
    )
    mcherry_images_list.append(mcherry_image)

    # ----- Channel 3: tdTomato (anatomical, PlanarMicroscopyStaticImage) -----
    data_tdtomato = np.random.rand(height, width).astype(np.float32)
    tdtomato_image = PlanarMicroscopyStaticImage(
        name=f"tdtomato_{depth}um",
        description=f"tdTomato anatomical image at {depth} µm — inhibitory (PV+) neurons",
        microscopy_rig=microscopy_rig,
        microscopy_channel=channel_tdtomato,
        planar_imaging_space=space_tdtomato,
        data=data_tdtomato,
    )
    tdtomato_images_list.append(tdtomato_image)

print(f"Created {len(gcamp_series_list)} GCaMP series, "
      f"{len(mcherry_images_list)} mCherry images, "
      f"{len(tdtomato_images_list)} tdTomato images.")

## 8. Build the multi-plane / multi-channel container hierarchy

Each `MultiPlaneMicroscopyContainer` groups all planes from a single channel.  
The `MultiChannelMicroscopyContainer` then groups all per-channel containers together.

In [ ]:
# One MultiPlaneMicroscopyContainer per channel
multi_plane_gcamp = MultiPlaneMicroscopyContainer(
    name="gcamp_planes",
    planar_microscopy_series=gcamp_series_list,
)

multi_plane_mcherry = MultiPlaneMicroscopyContainer(
    name="mcherry_planes",
    planar_microscopy_static_images=mcherry_images_list,
)

multi_plane_tdtomato = MultiPlaneMicroscopyContainer(
    name="tdtomato_planes",
    planar_microscopy_static_images=tdtomato_images_list,
)

# Top-level container: all channels combined
multi_channel_container = MultiChannelMicroscopyContainer(
    name="multi_plane_multi_channel_data",
    multi_plane_microscopy_containers=[multi_plane_gcamp, multi_plane_mcherry, multi_plane_tdtomato],
)

nwbfile.add_acquisition(multi_channel_container)
print("MultiChannelMicroscopyContainer added to acquisition.")

## 9. ROI segmentation on the GCaMP6f channel

Segmentation is performed independently on each focal plane of the functional channel.  
Summary images (mean and max projection) are stored alongside the ROI table.

In [ ]:
ophys_module = nwbfile.create_processing_module(
    name="ophys",
    description="Optical physiology processing module",
)

segmentation_list   = []
response_series_list = []

for depth, gcamp_series in zip(depths_in_um, gcamp_series_list):
    data = gcamp_series.data[:]  # shape: (frames, height, width)

    # Summary images
    mean_img = SummaryImage(
        name=f"mean_{depth}um",
        description=f"Mean intensity projection at {depth} µm",
        data=np.mean(data, axis=0),
    )
    max_img = SummaryImage(
        name=f"max_{depth}um",
        description=f"Maximum intensity projection at {depth} µm",
        data=np.max(data, axis=0),
    )

    # Segmentation table for this plane
    segmentation = PlanarSegmentation(
        name=f"rois_{depth}um",
        description=f"Manual ROI segmentation at {depth} µm",
        planar_imaging_space=gcamp_series.planar_imaging_space,
        summary_images=[mean_img, max_img],
    )

    # Add a few synthetic ROIs using image masks
    np.random.seed(42 + depths_in_um.index(depth))
    n_rois = 5
    for i in range(n_rois):
        cx = np.random.randint(20, height - 20)
        cy = np.random.randint(20, width - 20)
        roi_mask = np.zeros((height, width), dtype=bool)
        roi_mask[cx - 5:cx + 5, cy - 5:cy + 5] = True
        segmentation.add_roi(image_mask=roi_mask)

    segmentation_list.append(segmentation)

    # ROI region pointing to all ROIs in this table
    roi_region = segmentation.create_roi_table_region(
        description=f"All ROIs at {depth} µm",
        region=list(range(len(segmentation.id))),
    )

    # Extract fluorescence responses by averaging pixels within each ROI mask
    n_rois_actual = len(segmentation.id)
    responses = np.zeros((frames, n_rois_actual), dtype=np.float32)
    for i, roi_mask in enumerate(segmentation.image_mask[:]):
        responses[:, i] = data[:, roi_mask].mean(axis=1)

    response_series = MicroscopyResponseSeries(
        name=f"responses_{depth}um",
        description=f"Fluorescence responses (ΔF/F proxy) at {depth} µm",
        data=responses,
        rois=roi_region,
        unit="n.a.",
        rate=frame_rate_hz,
        starting_time=0.0,
        microscopy_series=gcamp_series,  # link back to the raw GCaMP series
    )
    response_series_list.append(response_series)

# Group all segmentations and responses into containers
segmentation_container = SegmentationContainer(
    name="plane_segmentations",
    segmentations=segmentation_list,
)
ophys_module.add(segmentation_container)

response_container = MicroscopyResponseSeriesContainer(
    name="plane_responses",
    microscopy_response_series=response_series_list,
)
ophys_module.add(response_container)

print(f"Segmentation done: {len(segmentation_list)} planes, "
      f"{len(segmentation_list[0].id)} ROIs each.")

## 10. Save the NWB file

In [ ]:
nwb_path = "multi_plane_multichannel_imaging.nwb"

with NWBHDF5IO(nwb_path, "w") as io:
    io.write(nwbfile)

print(f"NWB file saved to: {nwb_path}")

## 11. Read back and explore the data

Demonstrate how to navigate the container hierarchy after loading the file.

In [ ]:
with NWBHDF5IO(nwb_path, "r") as io:
    nwb_read = io.read()

    # ---- Navigate the acquisition hierarchy ----
    mc_container = nwb_read.acquisition["multi_plane_multi_channel_data"]
    print("Top-level container:", mc_container.name)
    print("  Nested MultiPlaneMicroscopyContainers:")
    for mp in mc_container.multi_plane_microscopy_containers.values():
        print(f"    {mp.name}")

    # ---- Access GCaMP functional data ----
    gcamp_container = mc_container.multi_plane_microscopy_containers["gcamp_planes"]
    print("\nGCaMP planes:")
    for series_name, series in gcamp_container.planar_microscopy_series.items():
        print(f"  {series_name}: shape = {series.data.shape}, rate = {series.rate} Hz")

    # ---- Access mCherry anatomical images ----
    mcherry_container = mc_container.multi_plane_microscopy_containers["mcherry_planes"]
    print("\nmCherry static images:")
    for img_name, img in mcherry_container.planar_microscopy_static_images.items():
        print(f"  {img_name}: shape = {img.data.shape}")

    # ---- Access segmentation and responses for depth 0 µm ----
    ophys = nwb_read.processing["ophys"]
    seg_container = ophys["plane_segmentations"]
    rois_0um = seg_container["rois_0um"]
    print(f"\nROIs at 0 µm: {len(rois_0um.id)} ROIs")
    print(f"  Image mask shape (first ROI): {rois_0um.image_mask[0].shape}")

    resp_container = ophys["plane_responses"]
    resp_0um = resp_container["responses_0um"]
    print(f"\nResponses at 0 µm: shape = {resp_0um.data.shape} (frames × ROIs)")

## 12. Quick visualisation

Plot the mean-projection images for all depths (GCaMP, mCherry, tdTomato) alongside the segmentation masks and fluorescence traces.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

with NWBHDF5IO(nwb_path, "r") as io:
    nwb_read = io.read()
    mc_container = nwb_read.acquisition["multi_plane_multi_channel_data"]
    ophys = nwb_read.processing["ophys"]
    seg_container = ophys["plane_segmentations"]
    resp_container = ophys["plane_responses"]

    n_depths = len(depths_in_um)
    fig, axes = plt.subplots(n_depths, 4, figsize=(16, 4 * n_depths))
    fig.suptitle("Multi-plane multi-channel two-photon imaging", fontsize=14, fontweight="bold")

    gcamp_mp  = mc_container.multi_plane_microscopy_containers["gcamp_planes"]
    mcherry_mp = mc_container.multi_plane_microscopy_containers["mcherry_planes"]
    tdtomato_mp = mc_container.multi_plane_microscopy_containers["tdtomato_planes"]

    for row, depth in enumerate(depths_in_um):
        ax_g, ax_m, ax_t, ax_r = axes[row]

        # GCaMP mean projection
        gcamp_data = gcamp_mp.planar_microscopy_series[f"gcamp_{depth}um"].data[:]
        ax_g.imshow(gcamp_data.mean(axis=0), cmap="Greens", interpolation="none")
        ax_g.set_title(f"GCaMP6f  {depth} µm")
        ax_g.axis("off")

        # mCherry static image
        mcherry_data = mcherry_mp.planar_microscopy_static_images[f"mcherry_{depth}um"].data[:]
        ax_m.imshow(mcherry_data, cmap="Reds", interpolation="none")
        ax_m.set_title(f"mCherry  {depth} µm")
        ax_m.axis("off")

        # tdTomato static image
        tdtomato_data = tdtomato_mp.planar_microscopy_static_images[f"tdtomato_{depth}um"].data[:]
        ax_t.imshow(tdtomato_data, cmap="Oranges", interpolation="none")
        ax_t.set_title(f"tdTomato  {depth} µm")
        ax_t.axis("off")

        # Fluorescence traces for this depth
        resp = resp_container[f"responses_{depth}um"].data[:]
        time = np.arange(resp.shape[0]) / frame_rate_hz
        for i in range(resp.shape[1]):
            ax_r.plot(time, resp[:, i] + i * 0.5, lw=0.6, label=f"ROI {i}")
        ax_r.set_xlabel("Time (s)")
        ax_r.set_ylabel("ΔF (a.u., offset)")
        ax_r.set_title(f"GCaMP responses  {depth} µm")
        ax_r.legend(fontsize=6, loc="upper right")

    plt.tight_layout()
    plt.show()